<a href="https://colab.research.google.com/github/pbanavara/notebooks/blob/main/Solve_Business_Problems_with_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Solve Business Problems with AI

## Objective
Develop a proof-of-concept application to intelligently process email order requests and customer inquiries for a fashion store. The system should accurately categorize emails as either product inquiries or order requests and generate appropriate responses using the product catalog information and current stock status.

You are encouraged to use AI assistants (like ChatGPT or Claude) and any IDE of your choice to develop your solution. Many modern IDEs (such as PyCharm, or Cursor) can work with Jupiter files directly.

## Task Description

### Inputs

Google Spreadsheet **[Document](https://docs.google.com/spreadsheets/d/14fKHsblfqZfWj3iAaM2oA51TlYfQlFT4WKo52fVaQ9U)** containing:

- **Products**: List of products with fields including product ID, name, category, stock amount, detailed description, and season.

- **Emails**: Sequential list of emails with fields such as email ID, subject, and body.

### Instructions

- Implement all requirements using advanced Large Language Models (LLMs) to handle complex tasks, process extensive data, and generate accurate outputs effectively.
- Use Retrieval-Augmented Generation (RAG) and vector store techniques where applicable to retrieve relevant information and generate responses.
- You are provided with a temporary OpenAI API key granting access to GPT-4o, which has a token quota. Use it wisely or use your own key if preferred.
- Address the requirements in the order listed. Review them in advance to develop a general implementation plan before starting.
- Your deliverables should include:
   - Code developed within this notebook.
   - A single spreadsheet containing results, organized across separate sheets.
   - Comments detailing your thought process.
- You may use additional libraries (e.g., langchain) to streamline the solution. Use libraries appropriately to align with best practices for AI and LLM tools.
- Use the most suitable AI techniques for each task. Note that solving tasks with traditional programming methods will not earn points, as this assessment evaluates your knowledge of LLM tools and best practices.

### Requirements

#### 1. Classify emails
    
Classify each email as either a _**"product inquiry"**_ or an _**"order request"**_. Ensure that the classification accurately reflects the intent of the email.

**Output**: Populate the **email-classification** sheet with columns: email ID, category.

#### 2. Process order requests
1.   Process orders
  - For each order request, verify product availability in stock.
  - If the order can be fulfilled, create a new order line with the status “created”.
  - If the order cannot be fulfilled due to insufficient stock, create a line with the status “out of stock” and include the requested quantity.
  - Update stock levels after processing each order.
  - Record each product request from the email.
  - **Output**: Populate the **order-status** sheet with columns: email ID, product ID, quantity, status (**_"created"_**, **_"out of stock"_**).

2.   Generate responses
  - Create response emails based on the order processing results:
      - If the order is fully processed, inform the customer and provide product details.
      - If the order cannot be fulfilled or is only partially fulfilled, explain the situation, specify the out-of-stock items, and suggest alternatives or options (e.g., waiting for restock).
  - Ensure the email tone is professional and production-ready.
  - **Output**: Populate the **order-response** sheet with columns: email ID, response.

#### 3. Handle product inquiry

Customers may ask general open questions.
  - Respond to product inquiries using relevant information from the product catalog.
  - Ensure your solution scales to handle a full catalog of over 100,000 products without exceeding token limits. Avoid including the entire catalog in the prompt.
  - **Output**: Populate the **inquiry-response** sheet with columns: email ID, response.

## Evaluation Criteria
- **Advanced AI Techniques**: The system should use Retrieval-Augmented Generation (RAG) and vector store techniques to retrieve relevant information from data sources and use it to respond to customer inquiries.
- **Tone Adaptation**: The AI should adapt its tone appropriately based on the context of the customer's inquiry. Responses should be informative and enhance the customer experience.
- **Code Completeness**: All functionalities outlined in the requirements must be fully implemented and operational as described.
- **Code Quality and Clarity**: The code should be well-organized, with clear logic and a structured approach. It should be easy to understand and maintain.
- **Presence of Expected Outputs**: All specified outputs must be correctly generated and saved in the appropriate sheets of the output spreadsheet. Ensure the format of each output matches the requirements—do not add extra columns or sheets.
- **Accuracy of Outputs**: The accuracy of the generated outputs is crucial and will significantly impact the evaluation of your submission.

We look forward to seeing your solution and your approach to solving real-world problems with AI technologies.

# Prerequisites

### Configure OpenAI API Key.

In [15]:
# Install the OpenAI Python package.
%pip install openai httpx==0.27.2

Note: you may need to restart the kernel to use updated packages.


**IMPORTANT: If you are going to use our custom API Key then make sure that you also use custom base URL as in example below. Otherwise it will not work.**

In [16]:
# Code example of OpenAI communication

from openai import OpenAI

client = OpenAI(
    # In order to use provided API key, make sure that models you create point to this custom base URL.
    base_url='https://47v4us7kyypinfb5lcligtc3x40ygqbs.lambda-url.us-east-1.on.aws/v1/',
    # The temporary API key giving access to ChatGPT 4o model. Quotas apply: you have 500'000 input and 500'000 output tokens, use them wisely ;)
    api_key='a0Bfv000001d3ThEAI'
)

completion = client.chat.completions.create(
  model="gpt-4o",
  messages=[
    {"role": "user", "content": "Hello!"}
  ]
)

print(completion.choices[0].message)

ChatCompletionMessage(content='Hello! How can I assist you today?', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None)


In [17]:
# Code example of reading input data

import pandas as pd
from IPython.display import display

def read_data_frame(document_id, sheet_name):
    export_link = f"https://docs.google.com/spreadsheets/d/{document_id}/gviz/tq?tqx=out:csv&sheet={sheet_name}"

    return  pd.read_csv(export_link)

document_id = '14fKHsblfqZfWj3iAaM2oA51TlYfQlFT4WKo52fVaQ9U'
products_df = read_data_frame(document_id, 'products')
emails_df = read_data_frame(document_id, 'emails')

# Display first 3 rows of each DataFrame
display(products_df.head(3))
display(emails_df.head(3))

,product_id,name,category,description,stock,seasons,price
0,RSG8901,Retro Sunglasses,Accessories,Transport yourself back in time with our retro...,1,"Spring, Summer",26.99
1,SWL2345,Sleek Wallet,Accessories,Keep your essentials organized and secure with...,5,All seasons,30.00
2,VSC6789,Versatile Scarf,Accessories,Add a touch of versatility to your wardrobe wi...,6,"Spring, Fall",23.00


,email_id,subject,message
0,E001,Leather Wallets,"Hi there, I want to order all the remaining LT..."
1,E002,Buy Vibrant Tote with noise,"Good morning, I'm looking to buy the VBT2345 V..."
2,E003,Need your help,"Hello, I need a new bag to carry my laptop and..."


# Task 1. Classify emails

In [18]:
# Let's classify the emails using OpenAI API using an appropriate prompt
def classify_email(email_content):
    prompt = f"""Classify the following email into one of the categories: 'order request', 'product inquiry'. Ensure that the classification is accurate and based on the intent in the email.
    Provide only the category as the response.

    Email:
    {email_content}
    """
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    message = response.choices[0].message.content.strip().lower()
    return message

# Use the emails_df DataFrame to classify each email and store the results in a new DataFrame
email_classification_df = emails_df.copy()
email_classification_df['category'] = email_classification_df['message'].apply(classify_email)

In [19]:
# Split the email_classification_df into order requests and product inquiries dataframes
order_requests_df = email_classification_df[
    email_classification_df["category"] == "order request"
]
product_inquiries_df = email_classification_df[
    email_classification_df["category"] == "product inquiry"
]


In [20]:
print(order_requests_df.head())

  email_id                               subject  \
0     E001                       Leather Wallets   
3     E004            Buy Infinity Scarves Order   
6     E007           Order for Beanies, Slippers   
7     E008  Ordering a Versatile Scarf-like item   
9     E010             Purchase Retro Sunglasses   

                                             message       category  
0  Hi there, I want to order all the remaining LT...  order request  
3  Hi, I'd like to order three to four SFT1098 In...  order request  
6  Hi, this is Liz. Please send me 5 CLF2109 Cabl...  order request  
7  Hello, I'd want to order one of your Versatile...  order request  
9  Hello, I would like to order 1 pair of RSG8901...  order request  


In [21]:
print(products_df.head())

  product_id                name     category  \
0    RSG8901    Retro Sunglasses  Accessories   
1    SWL2345        Sleek Wallet  Accessories   
2    VSC6789     Versatile Scarf  Accessories   
3    CSH1098          Cozy Shawl  Accessories   
4    CHN0987  Chunky Knit Beanie  Accessories   

                                         description  stock         seasons  \
0  Transport yourself back in time with our retro...      1  Spring, Summer   
1  Keep your essentials organized and secure with...      5     All seasons   
2  Add a touch of versatility to your wardrobe wi...      6    Spring, Fall   
3  Wrap yourself in comfort with our cozy shawl. ...      3    Fall, Winter   
4  Keep your head toasty with our chunky knit bea...      2    Fall, Winter   

   price  
0  26.99  
1  30.00  
2  23.00  
3  22.00  
4  22.00  


# Task 2. Process order requests

In [23]:
"""
Process the order requests and generate responses.
For each order request, verify product availability in stock.
If the order can be fulfilled, create a new order line with the status “created”.
If the order cannot be fulfilled due to insufficient stock, create a line with the status “out of stock” and include the requested quantity.
Update stock levels after processing each order.
Record each product request from the email.
Output: Populate the order-status sheet with columns: email ID, product ID, quantity, status ("created", "out of stock")
"""
order_status_df = pd.DataFrame(columns=['email_id', 'product_id', 'quantity', 'status'])
# Create a copy of products_df to track stock levels
current_stock_df = products_df.set_index('product_id').copy()
# We will use RAG to process the orders in the order_requests_df by using the product catalog in products_df. We will use OPenAI's tool calling feature and provide the product catalog lookup as a tool.
# First store the product catalog as a vector store using FAISS. For FAISS we will use product name and description as the text to embed. Then we will create an index over these embeddings. We then create a reverse index from the embeddings to product ID so that we can lookup product IDs from the embeddings.


In [24]:
%pip install faiss-cpu sentence-transformers

Note: you may need to restart the kernel to use updated packages.


In [26]:
import pandas as pd
import numpy as np
import faiss
from typing import List, Tuple, Dict
import json
import openai
import os


class ProductCatalogVectorStore:
    def __init__(
        self, use_openai: bool = False, openai_model: str = "text-embedding-3-small"
    ):
        """
        Initialize the vector store with either sentence transformers or OpenAI embeddings.

        Args:
            use_openai: Whether to use OpenAI embeddings instead of sentence-transformers
            openai_model: OpenAI embedding model to use
        """
        self.use_openai = use_openai
        self.openai_model = openai_model

        if use_openai:
            # Ensure OpenAI API key is set
            self.client = OpenAI(
                base_url="https://47v4us7kyypinfb5lcligtc3x40ygqbs.lambda-url.us-east-1.on.aws/v1/",
                api_key="a0Bfv000001d3ThEAI",
            )
        else:
            try:
                from sentence_transformers import SentenceTransformer

                self.model = SentenceTransformer("all-MiniLM-L6-v2")
            except ImportError as e:
                print(f"❌ Error loading sentence-transformers: {e}")
                print("💡 Try: conda install gfortran")
                print("💡 Or set use_openai=True to use OpenAI embeddings")
                raise

        self.index = None
        self.product_id_map = {}  # Maps FAISS index -> product_id
        self.products_df = None
        self.embeddings = None

    def build_index(self, products_df: pd.DataFrame):
        """
        Build FAISS index from product catalog dataframe.

        Args:
            products_df: DataFrame with columns: product_id, name, description, category, stock, seasons
        """
        self.products_df = products_df.copy()

        # Combine name and description for better search results
        texts_to_embed = []
        for _, row in products_df.iterrows():
            # Create rich text combining name, description, and category
            text = f"{row['name']} - {row['description']} Category: {row['category']}"
            texts_to_embed.append(text)

        print(f"Generating embeddings for {len(texts_to_embed)} products...")

        # Generate embeddings
        if self.use_openai:
            self.embeddings = self._get_openai_embeddings(texts_to_embed)
        else:
            self.embeddings = self.model.encode(texts_to_embed)

        # Convert to float32 and ensure C-contiguous for FAISS
        self.embeddings = np.ascontiguousarray(self.embeddings, dtype=np.float32)

        # Build FAISS index
        dimension = self.embeddings.shape[1]
        self.index = faiss.IndexFlatIP(dimension)  # Inner Product for cosine similarity

        # Normalize embeddings for cosine similarity
        faiss.normalize_L2(self.embeddings)

        # Add embeddings to index
        self.index.add(self.embeddings)

        # Create reverse mapping from FAISS index to product_id
        self.product_id_map = {
            i: products_df.iloc[i]["product_id"] for i in range(len(products_df))
        }

        print(f"✅ FAISS index built with {self.index.ntotal} products")

    def search(self, query: str, k: int = 5) -> List[Dict]:
        """
        Search for similar products using semantic similarity.

        Args:
            query: Search query string
            k: Number of results to return

        Returns:
            List of dictionaries containing product info and similarity scores
        """
        if self.index is None:
            raise ValueError("Index not built. Call build_index() first.")

        # Generate query embedding
        if self.use_openai:
            query_embedding = self._get_openai_embeddings([query])
        else:
            query_embedding = self.model.encode([query])

        # Convert to float32 and ensure C-contiguous for FAISS
        query_embedding = np.ascontiguousarray(query_embedding, dtype=np.float32)
        faiss.normalize_L2(query_embedding)

        # Search
        scores, indices = self.index.search(query_embedding, k)

        results = []
        for score, idx in zip(scores[0], indices[0]):
            if idx == -1:  # FAISS returns -1 for no match
                continue

            product_id = self.product_id_map[idx]
            product_info = (
                self.products_df[self.products_df["product_id"] == product_id]
                .iloc[0]
                .to_dict()
            )

            results.append(
                {
                    "product_id": product_id,
                    "similarity_score": float(score),
                    "product_info": product_info,
                }
            )

        return results

    def get_product_by_id(self, product_id: str) -> Dict:
        """
        Get product information by product ID.

        Args:
            product_id: Product ID to lookup

        Returns:
            Dictionary containing product information
        """
        product = self.products_df[self.products_df["product_id"] == product_id]

        if product.empty:
            return None

        return product.iloc[0].to_dict()

    def _get_openai_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Get embeddings from OpenAI API.

        Args:
            texts: List of texts to embed

        Returns:
            Numpy array of embeddings (float32, C-contiguous)
        """
        try:
            response = self.client.embeddings.create(
                model=self.openai_model, input=texts
            )
            # Convert to numpy array with proper dtype and memory layout
            embeddings = np.array(
                [item.embedding for item in response.data], dtype=np.float32
            )
            return np.ascontiguousarray(embeddings)
        except Exception as e:
            print(f"❌ Error getting OpenAI embeddings: {e}")
            raise


In [27]:
vector_store = ProductCatalogVectorStore(use_openai=True)
vector_store.build_index(products_df)

Generating embeddings for 99 products...
✅ FAISS index built with 99 products


In [28]:
# Test 
results = vector_store.search("winter accessories", k=3)
print(f"\n📦 Search: 'winter accessories'")
for result in results:
    product = result["product_info"]
    print(f"  • {product['name']} (Score: {result['similarity_score']:.3f})")


📦 Search: 'winter accessories'
  • Earmuffs (Score: 0.542)
  • Infinity Scarf (Score: 0.524)
  • Snow Boots (Score: 0.481)


In [101]:
# Use OpeAI tool calling to integrate the product catalog lookup into the LLM workflow
def product_catalog_lookup(query: str, max_results: int = 1) -> str:
    """
    Tool function for OpenAI function calling to search product catalog.

    Args:
        query: Search query for products
        max_results: Maximum number of results to return

    Returns:
        JSON string containing search results
    """
    results = vector_store.search(query, k=max_results)

    # Format results for LLM consumption
    formatted_results = []
    for result in results:
        product = result["product_info"]
        formatted_results.append(
            {
                "product_id": product["product_id"],
                "name": product["name"],
                "category": product["category"],
                "description": product["description"],
                "stock": product["stock"],
                "seasons": product["seasons"],
                "relevance_score": round(result["similarity_score"], 3),
            }
        )

    return formatted_results

PRODUCT_LOOKUP_TOOL = {
    "type": "function",
    "function": {
        "name": "product_catalog_lookup",
        "description": "Search the product catalog for items matching a query. Use this to find products based on descriptions, categories, or names.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Search query describing the product you're looking for",
                },
                "max_results": {
                    "type": "integer",
                    "description": "Maximum number of results to return (default: 3)",
                    "default": 3,
                },
            },
            "required": ["query"],
        },
    },
}


In [60]:

# Use the order requests dataframe to test the tool calling
# Use the first 2 rows of order_requests_df for testing
order_tool_messages = []
for _, row in order_requests_df.iterrows():
    email_id = row["email_id"]
    email_content = row["message"]

    # Create a prompt to extract product details from the email
    client = OpenAI(
        base_url='https://47v4us7kyypinfb5lcligtc3x40ygqbs.lambda-url.us-east-1.on.aws/v1/',
        api_key='a0Bfv000001d3ThEAI'
    )
    response = client.chat.completions.create(
        model="gpt-4o",  # or "gpt-4.1", "gpt-4o-mini", etc.
        messages=[{"role": "user", "content": email_content}],
        tools=[PRODUCT_LOOKUP_TOOL],
        tool_choice="auto",
    )

    message = response.choices[0].message
    if message.tool_calls:
        for tc in message.tool_calls:
            if tc.type == "function" and tc.function.name == "product_catalog_lookup":
                args = json.loads(tc.function.arguments or "{}")
                query = args.get("query")
                max_results = args.get("max_results", 1)

                # Your actual tool execution
                search_results = product_catalog_lookup(query, max_results)
                # 3) Return tool output back to the model
                order_tool_messages.append(
                    {
                        "role": "tool",
                        "enail_id": email_id,
                        "tool_call_id": tc.id,  # <-- important
                        "content": json.dumps(search_results),  # string content
                    }
                )

In [ ]:
#Prepare the order status output with the following fields email ID, product ID, quantity, status ("created", "out of stock")

rows_data = []  # Build list first instead of empty DataFrame

for row in order_tool_messages:
    try:
        email_id = row["enail_id"]  # Fixed typo: was 'enail_id'
        tool_output = json.loads(json.loads(row["content"]))
        
        if tool_output:
            product_id = tool_output[0].get("product_id")
            quantity = tool_output[0].get("stock", 0)  # Safe access with default
            status = "created" if quantity > 0 else "out of stock"

            rows_data.append(
                {
                    "email_id": email_id,
                    "product_id": product_id,
                    "quantity": quantity,
                    "status": status,
                }
            )
    except (json.JSONDecodeError, KeyError, IndexError) as e:
        print(f"❌ Error processing row: {e}")
        continue

print(rows_data)
# Create DataFrame in one go (MUCH faster)
order_status_df = pd.DataFrame(rows_data)

[{'email_id': 'E001', 'product_id': 'LTH0976', 'quantity': 4, 'status': 'created'}, {'email_id': 'E004', 'product_id': 'SFT1098', 'quantity': 8, 'status': 'created'}, {'email_id': 'E007', 'product_id': 'CLF2109', 'quantity': 2, 'status': 'created'}, {'email_id': 'E007', 'product_id': 'FZZ1098', 'quantity': 2, 'status': 'created'}, {'email_id': 'E008', 'product_id': 'VSC6789', 'quantity': 6, 'status': 'created'}, {'email_id': 'E010', 'product_id': 'RSG8901', 'quantity': 1, 'status': 'created'}, {'email_id': 'E013', 'product_id': 'SLD7654', 'quantity': 3, 'status': 'created'}, {'email_id': 'E014', 'product_id': 'SWL2345', 'quantity': 5, 'status': 'created'}, {'email_id': 'E017', 'product_id': 'BMX2345', 'quantity': 1, 'status': 'created'}, {'email_id': 'E018', 'product_id': 'RSG8901', 'quantity': 1, 'status': 'created'}, {'email_id': 'E019', 'product_id': 'CBT8901', 'quantity': 2, 'status': 'created'}, {'email_id': 'E019', 'product_id': 'FZZ1098', 'quantity': 2, 'status': 'created'}, {'e

In [87]:
order_status_df.head()

,email_id,product_id,quantity,status
0,E001,LTH0976,4,created
1,E004,SFT1098,8,created
2,E007,CLF2109,2,created
3,E007,FZZ1098,2,created
4,E008,VSC6789,6,created


### Store First classification output

In [ ]:
import gspread
from google.oauth2.service_account import Credentials

SCOPES = ["https://www.googleapis.com/auth/spreadsheets"]
creds = Credentials.from_service_account_file("service_account.json", scopes=SCOPES)
gc = gspread.authorize(creds)

In [35]:
spreadsheet = gc.open_by_key("1lOaearbK7ujOuevBQofG5MdjiudYJlqPpyUiSjwOkws")
email_classification_sheet = spreadsheet.add_worksheet(
    title="email-classification", rows=50, cols=2
)


In [ ]:
email_classification_df_out = email_classification_df[["email_id", "category"]]

,email_id,category
0,E001,order request
1,E002,product inquiry
2,E003,product inquiry
3,E004,order request
4,E005,product inquiry
5,E006,product inquiry
6,E007,order request
7,E008,order request
8,E009,product inquiry
9,E010,order request


In [48]:
from gspread_dataframe import set_with_dataframe

def gspread_dataframe(worksheet, df):
    """Method 2: Using gspread_dataframe library (easiest)."""

    print("\n📊 Method 2: Using gspread_dataframe")

    # Clear the worksheet first (optional)
    worksheet.clear()

    # ✅ Super simple with gspread_dataframe
    set_with_dataframe(worksheet, df, include_index=False, include_column_header=True)
    print("   ✅ gspread_dataframe method completed")

gspread_dataframe(email_classification_sheet, email_classification_df_out)


📊 Method 2: Using gspread_dataframe
   ✅ gspread_dataframe method completed


### Store order output

In [116]:
# Does not check for existing sheet, will error if it exists
order_status_sheet = spreadsheet.add_worksheet(
    title="order-status", rows=50, cols=5
)
gspread_dataframe(order_status_sheet, order_status_df)


📊 Method 2: Using gspread_dataframe
   ✅ gspread_dataframe method completed


# Task 3. Handle product inquiry

In [ ]:
# Use the same toolchain but to answer product inquiries by using different prompts
inquiry_tool_messages = []
for _, row in product_inquiries_df.iterrows():
    email_id = row["email_id"]
    email_content = row["message"]

    prompt = (
        "You are a shopping assistant agent. "
        f"For email {email_id} with content: {email_content} "
        "call the product_catalog_lookup tool to find relevant products. "
        "When you get the tool response (JSON), parse it and reply in EXACTLY this format:\n"
        "“We have the products you requested. Available: <product_id> with the following description <product_description> and quantity (<stock>), …”\n"
        "If no results, reply: “No matching products found.” Do not echo raw JSON."
    )

    # ---------- 1) First call: let the model emit tool_calls ----------
    resp1 = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}],
        tools=[PRODUCT_LOOKUP_TOOL],
        tool_choice="auto",
    )

    first_msg = resp1.choices[0].message

    # Build a plain-dict assistant message with tool_calls (DO NOT pass SDK object directly)
    assistant_toolcall_msg = None
    tool_outputs = []

    if getattr(first_msg, "tool_calls", None):
        # Convert each tool_call into a plain dict; ensure arguments is a STRING
        tool_calls_plain = []
        for tc in first_msg.tool_calls:
            # tc.function.arguments should already be a JSON string; if not, dump it.
            args_str = tc.function.arguments
            if not isinstance(args_str, str):
                args_str = json.dumps(args_str)

            tool_calls_plain.append(
                {
                    "id": tc.id,
                    "type": "function",
                    "function": {
                        "name": tc.function.name,
                        "arguments": args_str,
                    },
                }
            )

        assistant_toolcall_msg = {
            "role": "assistant",
            "content": "",  # MUST be a string ("" is fine)
            "tool_calls": tool_calls_plain,
        }

        # ---------- Run each requested tool and collect tool messages ----------
        for tc in first_msg.tool_calls:
            if tc.type == "function" and tc.function.name == "product_catalog_lookup":
                try:
                    args = json.loads(tc.function.arguments or "{}")
                except Exception:
                    # As a fallback, try to coerce; but better to fail fast
                    args = {}

                query = args.get("query") or ""
                max_results = args.get("max_results", 3)

                # Execute your Python tool (return must be JSON STRING)
                search_results_json = product_catalog_lookup(query, max_results)
                if not isinstance(search_results_json, str):
                    # Ensure string for API
                    search_results_json = json.dumps(search_results_json)

                tool_outputs.append(
                    {
                        "role": "tool",
                        "tool_call_id": tc.id,  # REQUIRED
                        "content": search_results_json,  # MUST be a string
                    }
                )

    # ---------- 2) Second call: feed user msg + assistant toolcall msg + tool outputs ----------
    # If no tool call happened, you can skip second call (or just print a fallback)
    if assistant_toolcall_msg and tool_outputs:
        final_messages = [
            {"role": "user", "content": prompt},
            assistant_toolcall_msg,
            *tool_outputs,
        ]

        resp2 = client.chat.completions.create(
            model="gpt-4o",
            messages=final_messages,
        )

        final_text = resp2.choices[0].message.content or ""
        inquiry_tool_messages.append(
            {
                "email_id": email_id,
                "response": final_text,
            }
        )

    else:
        # Fallback: no tool call — either prompt was off, or the model refused.
        # You can choose to log/handle this deterministically.
        print(f"Email {email_id} → No matching products found.")
        inquiry_tool_messages.append(
            {
                "email_id": email_id,
                "response": "Sorry, no matching products found",
            }
        ) 

In [107]:
inquiry_tool_messages[0]

{'email_id': 'E002',
 'response': "We have the products you requested. Available: VBT2345 Add a pop of color to your everyday carry with our vibrant tote bag. Spacious and stylish, it's the perfect companion for running errands or carrying your essentials. The vibrant hue is sure to turn heads. (qty 4)"}

In [ ]:
# Build the inquiry response dataframe
inquiry_data = []  # Build list first instead of empty DataFrame

for row in inquiry_tool_messages:
    try:
        email_id = row["email_id"]  # Fixed typo: was 'enail_id'
        response_text = row["response"]
        inquiry_data.append(
            {
                "email_id": email_id,
                "response": response_text,
            }
        )
    except (json.JSONDecodeError, KeyError, IndexError) as e:
        print(f"❌ Error processing row: {e}")
        continue

# Create DataFrame in one go (MUCH faster)
inquiry_response_df = pd.DataFrame(inquiry_data)


[{'email_id': 'E002', 'response': "We have the products you requested. Available: VBT2345 with the following description Add a pop of color to your everyday carry with our vibrant tote bag. Spacious and stylish, it's the perfect companion for running errands or carrying your essentials. The vibrant hue is sure to turn heads. and quantity (4)."}, {'email_id': 'E003', 'response': 'We have the products you requested. Available: LTH1098 with the following description: Upgrade your daily carry with our leather backpack. Crafted from premium leather, this stylish backpack features multiple compartments, a padded laptop sleeve, and adjustable straps for a comfortable fit. Perfect for work, travel, or everyday use. and quantity (7), LTH5432 with the following description: Elevate your everyday carry with our leather tote bag. Crafted from premium, full-grain leather, this bag features a spacious interior, multiple pockets, and sturdy handles. Perfect for work, travel, or running errands in sty

In [ ]:
# Write the inqury_response_df to the order-response sheet
inquiry_response_sheet = spreadsheet.add_worksheet(
    title="order-response", rows=50, cols=2
)
gspread_dataframe(inquiry_response_sheet, inquiry_response_df)


📊 Method 2: Using gspread_dataframe
   ✅ gspread_dataframe method completed
